In [1]:
# Implementation
import torch
import torchvision
from torch import nn
from torchvision import transforms
import torch.nn.functional as F

In [2]:
# Define SimCLR
resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])

class SimCLR(nn.Module):
    def __init__(self, backbone, feature_dim = 512, out_dim = 128):
        super().__init__()
        self.backbone = backbone

        self.projector = nn.Sequential(
            # MLP with one hidden layer
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, out_dim)
        )

    def forward(self, x):
        h = self.backbone(x).flatten(1)
        z = self.projector(h)
        return z

model = SimCLR(backbone)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("")

In [3]:
# Data Augmentation
# size is equal to CIFAR10
img_size = 32

SimCLR_transforms = transforms.Compose([
    # Random crop with resize
    transforms.RandomResizedCrop(img_size),
    # Random flip
    transforms.RandomHorizontalFlip(p = 0.5),
    # Color distortion 
    transforms.RandomApply([transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)], p = 0.8),
    # Color drop                
    transforms.RandomGrayscale(p = 0.2),

    #Ready to feed model!
    transforms.ToTensor()
])

# Define a transforms wrapper to get two different data augmentation
class SimCLR_transforms_wrapper:
    def __init__(self, transforms):
        self.transforms = transforms

    # To fit Pytorch
    def __call__(self, image):
        x_i = self.transforms(image)
        x_j = self.transforms(image)
        return x_i, x_j

dataset = torchvision.datasets.CIFAR10(
    "datasets/cifar10", download = True, transform = SimCLR_transforms_wrapper(SimCLR_transforms)
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=True
)

Files already downloaded and verified


In [4]:
# Loss function
# Here I use Alignment and Uniformity as contrastive loss
class AlignUniformLoss(nn.Module):
    def __init__(self, alpha = 2, t = 2, lam = 1):
        super().__init__()
        self.alpha = alpha
        self.t = t
        self.lam = lam
        
    def lalign(self, x, y, alpha):
        return (x - y).norm(dim = 1).pow(alpha).mean()
    def lunif(self, x, t):
        sq_pdist = torch.pdist(x, p=2).pow(2)
        return sq_pdist.mul(-t).exp().mean().log()

    def forward(self, z_i, z_j):
        z_i = F.normalize(z_i, dim = 1)
        z_j = F.normalize(z_j, dim = 1)
        return self.lalign(z_i, z_j, self.alpha) + self.lam * (self.lunif(z_i, self.t) + self.lunif(z_j, self.t)) / 2

criterion = AlignUniformLoss()

In [5]:
# Optimizer using SGD not LARS
optimizer = torch.optim.SGD(model.parameters(), lr=0.06)

In [ ]:
#Training process
print("Starting Training")
num_epoch = 10
for epoch in range(num_epoch):
    total_loss = 0
    
    for (x_i, x_j),_ in dataloader:
        x_i = x_i.to(device)
        x_j = x_j.to(device)
        
        z_i = model(x_i)
        z_j = model(x_j)
    
        loss = criterion(z_i, z_j)
    
        total_loss += loss.item()
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    avg_loss = total_loss / len(dataloader)
    print(f"epoch: {epoch:>02}, loss: {avg_loss:.5f}")

Starting Training
epoch: 00, loss: -1.90351
epoch: 01, loss: -2.13922


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import torch, torchvision
from torchvision import transforms

eval_tf = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.CIFAR10("datasets/cifar10", train=True, transform=eval_tf)
test_set = torchvision.datasets.CIFAR10("datasets/cifar10", train=False, transform=eval_tf)

@torch.no_grad()
def get_feats(dataset):
    loader = torch.utils.data.DataLoader(dataset, batch_size=256, num_workers=4)
    model.backbone.eval()
    feats, labels = [], []
    for x, y in loader:
        f = model.backbone(x.to(device)).flatten(1).cpu()
        feats.append(f); labels.append(y)
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()

Xtr, ytr = get_feats(train_set)
Xte, yte = get_feats(test_set)

knn = KNeighborsClassifier(n_neighbors=20, metric="cosine")
knn.fit(Xtr, ytr)
print(f"KNN Accuracy: {knn.score(Xte, yte)*100:.2f}%")